<a href="https://colab.research.google.com/github/sandyreda898-oss/Smart-Agricultural-Water-Management/blob/main/weather_soil_water_modeling/notebooks/06_irrigation_requirement_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

repo_path = "/content/Smart-Agricultural-Water-Management"

if not os.path.exists(repo_path):
    !git clone https://github.com/sandyreda898-oss/Smart-Agricultural-Water-Management.git
else:
    print("Repository already exists. Using the current copy.")

Cloning into 'Smart-Agricultural-Water-Management'...
remote: Enumerating objects: 723, done.
remote: Counting objects: 100% (161/161), done.
remote: Compressing objects: 100% (152/152), done.
remote: Total 723 (delta 56), reused 0 (delta 0), pack-reused 562 (from 1)
Receiving objects: 100% (723/723), 165.14 MiB | 19.43 MiB/s, done.
Resolving deltas: 100% (295/295), done.
Updating files: 100% (452/452), done.


In [2]:
import pandas as pd
import numpy as np
import os
import glob

base = (
    "/content/Smart-Agricultural-Water-Management/"
)

processed_path = (
    base +
    "weather_soil_water_modeling/data/processed/"
)

reference_path = (
    base +
    "weather_soil_water_modeling/data/reference/"
)

raw_path = (
    base +
    "AgriWater/AgriWater_Clean_Data/data/table_exports/"
)

In [3]:
etc = pd.read_csv(
    processed_path + "etc_daily.csv.gz",
    parse_dates=["obs_date"]
)

weather = pd.read_csv(
    processed_path + "weather_daily_clean.csv.gz",
    parse_dates=["obs_date"]
)

soil = pd.read_csv(
    processed_path + "soil_clean.csv"
)

crop_water = pd.read_csv(
    reference_path + "crop_water_parameters_reference.csv"
)

print("ETc:", etc.shape)
print("Weather:", weather.shape)
print("Soil:", soil.shape)
print("Crop water parameters:", crop_water.shape)

ETc: (39900, 48)
Weather: (136002, 17)
Soil: (36, 28)
Crop water parameters: (10, 15)


In [4]:
print("ETc columns:")
print(etc.columns.tolist())

print("\nSoil texture values:")
print(
    soil["texture_usda"]
    .value_counts(dropna=False)
)

print("\nCrop water parameters:")
display(
    crop_water[
        [
            "growth_stage",
            "parameter_name",
            "parameter_value",
            "value_min",
            "value_max",
            "unit",
            "data_status"
        ]
    ]
)

ETc columns:
['point_id', 'obs_date', 'ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'PS', 'RH2M', 'T2M', 'T2M_MAX', 'T2M_MIN', 'WS2M', 'latitude', 'longitude', 'district_id', 'spatial_support', 'et0_inputs_complete', 'rainfall_available', 'rainfall_review_flag', 'day_of_year', 'latitude_rad', 'es_tmax', 'es_tmin', 'es', 'ea', 'vpd', 'delta', 'gamma', 'inverse_relative_distance', 'solar_declination', 'sunset_hour_angle', 'Ra', 'Rs', 'Rso', 'rs_rso_raw', 'radiation_ratio_review_flag', 'rs_rso', 'Rns', 'Rnl', 'Rn', 'ET0_mm_day', 'season_year', 'start_date', 'end_date', 'day_in_season', 'growth_stage', 'kc_low', 'kc_high', 'ETc_low_mm_day', 'ETc_high_mm_day']

Soil texture values:
texture_usda
10.0    11
5.0     10
3.0      8
9.0      7
Name: count, dtype: int64

Crop water parameters:


,growth_stage,parameter_name,parameter_value,value_min,value_max,unit,data_status
0,initial,Kc,1.05,1.05,1.05,dimensionless,FAO_REFERENCE_NOT_LOCALLY_CALIBRATED
1,mid_season,Kc,1.20,1.20,1.20,dimensionless,FAO_REFERENCE_NOT_LOCALLY_CALIBRATED
2,late_season,Kc,NaN,0.60,0.90,dimensionless,FAO_REFERENCE_RANGE_NOT_LOCALLY_CALIBRATED
3,land_preparation,soil_saturation_requirement,200.00,200.00,200.00,mm,LITERATURE_ASSUMPTION_REQUIRES_LOCAL_VALIDATION
4,growing_season,seepage_percolation_rate,NaN,4.00,8.00,mm/day,LITERATURE_RANGE_REQUIRES_LOCAL_VALIDATION
5,establishment,water_layer_establishment,100.00,100.00,100.00,mm,LITERATURE_ASSUMPTION_REQUIRES_LOCAL_VALIDATION
6,initial,stage_length,30.00,30.00,30.00,days,FAO_REFERENCE_NOT_LOCALLY_CALIBRATED
7,development,stage_length,30.00,30.00,30.00,days,FAO_REFERENCE_NOT_LOCALLY_CALIBRATED
8,mid_season,stage_length,60.00,60.00,60.00,days,FAO_REFERENCE_NOT_LOCALLY_CALIBRATED
9,late_season,stage_length,30.00,30.00,30.00,days,FAO_REFERENCE_NOT_LOCALLY_CALIBRATED


In [5]:
rainfall_daily = weather[
    [
        "point_id",
        "obs_date",
        "PRECTOTCORR"
    ]
].copy()

if "PRECTOTCORR" in etc.columns:
    etc = etc.drop(
        columns=["PRECTOTCORR"]
    )

etc = etc.merge(
    rainfall_daily,
    on=["point_id", "obs_date"],
    how="left",
    validate="many_to_one"
)

print(
    "Missing rainfall after merge:",
    etc["PRECTOTCORR"].isna().sum()
)

print(
    "Total rice-season rows:",
    len(etc)
)

Missing rainfall after merge: 0
Total rice-season rows: 39900


In [6]:
# Extract the rice-water parameters
sat_row = crop_water[
    crop_water["parameter_name"]
    == "soil_saturation_requirement"
].iloc[0]

perc_row = crop_water[
    crop_water["parameter_name"]
    == "seepage_percolation_rate"
].iloc[0]

wl_row = crop_water[
    crop_water["parameter_name"]
    == "water_layer_establishment"
].iloc[0]

SAT_mm = float(
    sat_row["parameter_value"]
)

PERC_low_mm_day = float(
    perc_row["value_min"]
)

PERC_high_mm_day = float(
    perc_row["value_max"]
)

WL_mm = float(
    wl_row["parameter_value"]
)

print("SAT:", SAT_mm, "mm")
print(
    "PERC range:",
    PERC_low_mm_day,
    "to",
    PERC_high_mm_day,
    "mm/day"
)
print("WL:", WL_mm, "mm")

SAT: 200.0 mm
PERC range: 4.0 to 8.0 mm/day
WL: 100.0 mm


In [7]:
# Monthly rainfall
rain = weather[
    [
        "point_id",
        "obs_date",
        "PRECTOTCORR"
    ]
].copy()

rain["year"] = (
    rain["obs_date"].dt.year
)

rain["month"] = (
    rain["obs_date"].dt.month
)

monthly_rain = (
    rain.groupby(
        [
            "point_id",
            "year",
            "month"
        ],
        as_index=False
    )
    .agg(
        rainfall_month_mm=(
            "PRECTOTCORR",
            "sum"
        )
    )
)

monthly_rain.head()

,point_id,year,month,rainfall_month_mm
0,grid_0001,2017,1,6.84
1,grid_0001,2017,2,8.34
2,grid_0001,2017,3,0.41
3,grid_0001,2017,4,80.24
4,grid_0001,2017,5,0.05


In [8]:
# Calculate monthly effective rainfall
P = monthly_rain[
    "rainfall_month_mm"
]

monthly_rain[
    "effective_rain_month_mm"
] = np.where(
    P > 75,
    0.8 * P - 25,
    0.6 * P - 10
)

monthly_rain[
    "effective_rain_month_mm"
] = monthly_rain[
    "effective_rain_month_mm"
].clip(lower=0)

monthly_rain[
    [
        "rainfall_month_mm",
        "effective_rain_month_mm"
    ]
].describe()

,rainfall_month_mm,effective_rain_month_mm
count,4484.000000,4484.000000
mean,21.903918,10.467323
std,60.378283,43.749702
min,0.000000,0.000000
25%,0.000000,0.000000
50%,2.980000,0.000000
75%,20.380000,2.228000
max,628.080000,477.464000


In [9]:
monthly_rain[
    "effective_rain_fraction"
] = np.where(
    monthly_rain[
        "rainfall_month_mm"
    ] > 0,

    monthly_rain[
        "effective_rain_month_mm"
    ]
    /
    monthly_rain[
        "rainfall_month_mm"
    ],

    0
)

In [10]:
rain = rain.merge(
    monthly_rain,
    on=[
        "point_id",
        "year",
        "month"
    ],
    how="left",
    validate="many_to_one"
)

rain[
    "effective_rain_daily_mm"
] = (
    rain["PRECTOTCORR"]
    *
    rain["effective_rain_fraction"]
)

In [11]:
etc = etc.merge(
    rain[
        [
            "point_id",
            "obs_date",
            "effective_rain_daily_mm"
        ]
    ],
    on=[
        "point_id",
        "obs_date"
    ],
    how="left",
    validate="many_to_one"
)

print(
    "Missing effective rainfall:",
    etc[
        "effective_rain_daily_mm"
    ].isna().sum()
)

Missing effective rainfall: 0


In [12]:
# Seasonal totals
irrigation = (
    etc.groupby(
        [
            "point_id",
            "season_year"
        ],
        as_index=False
    )
    .agg(
        season_days=(
            "obs_date",
            "count"
        ),

        ETc_low_mm=(
            "ETc_low_mm_day",
            "sum"
        ),

        ETc_high_mm=(
            "ETc_high_mm_day",
            "sum"
        ),

        rainfall_total_mm=(
            "PRECTOTCORR",
            "sum"
        ),

        effective_rain_mm=(
            "effective_rain_daily_mm",
            "sum"
        )
    )
)

irrigation.head()

,point_id,season_year,season_days,ETc_low_mm,ETc_high_mm,rainfall_total_mm,effective_rain_mm
0,grid_0001,2017,150,1202.974001,1228.196566,62.43,15.155915
1,grid_0001,2018,150,1203.716044,1230.534286,3.07,0.000000
2,grid_0001,2019,150,1224.860108,1250.426308,0.00,0.000000
3,grid_0001,2020,150,1197.470565,1224.723246,0.07,0.000000
4,grid_0001,2021,150,1231.234540,1258.247982,0.00,0.000000


In [14]:
# Add rice-specific water needs
irrigation[
    "SAT_mm"
] = SAT_mm

irrigation[
    "WL_mm"
] = WL_mm

irrigation[
    "PERC_low_mm"
] = (
    irrigation["season_days"]
    *
    PERC_low_mm_day
)

irrigation[
    "PERC_high_mm"
] = (
    irrigation["season_days"]
    *
    PERC_high_mm_day
)

In [15]:
# Net irrigation requirement
irrigation[
    "net_irrigation_low_mm"
] = (
    irrigation["ETc_low_mm"]
    +
    irrigation["SAT_mm"]
    +
    irrigation["PERC_low_mm"]
    +
    irrigation["WL_mm"]
    -
    irrigation["effective_rain_mm"]
)

irrigation[
    "net_irrigation_high_mm"
] = (
    irrigation["ETc_high_mm"]
    +
    irrigation["SAT_mm"]
    +
    irrigation["PERC_high_mm"]
    +
    irrigation["WL_mm"]
    -
    irrigation["effective_rain_mm"]
)

irrigation[
    "net_irrigation_low_mm"
] = irrigation[
    "net_irrigation_low_mm"
].clip(lower=0)

irrigation[
    "net_irrigation_high_mm"
] = irrigation[
    "net_irrigation_high_mm"
].clip(lower=0)

In [16]:
# Convert to m³ per feddan
irrigation[
    "net_low_m3_per_feddan"
] = (
    irrigation[
        "net_irrigation_low_mm"
    ]
    * 4.2
)

irrigation[
    "net_high_m3_per_feddan"
] = (
    irrigation[
        "net_irrigation_high_mm"
    ]
    * 4.2
)

irrigation[
    [
        "net_irrigation_low_mm",
        "net_irrigation_high_mm",
        "net_low_m3_per_feddan",
        "net_high_m3_per_feddan"
    ]
].describe()

,net_irrigation_low_mm,net_irrigation_high_mm,net_low_m3_per_feddan,net_high_m3_per_feddan
count,266.000000,266.000000,266.000000,266.000000
mean,2012.476083,2637.903679,8452.399547,11079.195450
std,84.630912,85.533640,355.449831,359.241290
min,1912.330225,2536.592319,8031.786947,10653.687739
25%,1938.254349,2562.813561,8140.668265,10763.816957
50%,1963.440367,2589.242599,8246.449542,10874.818917
75%,2103.676894,2730.486909,8835.442957,11468.045017
max,2134.911432,2761.387850,8966.628016,11597.828971


In [17]:
all_raw_files = sorted(
    os.path.basename(path)
    for path in glob.glob(
        raw_path + "*"
    )
)

keywords = [
    "water",
    "irrig",
    "scenario",
    "trial",
    "experiment",
    "observation",
    "environment"
]

validation_files = [
    name
    for name in all_raw_files
    if any(
        word in name.lower()
        for word in keywords
    )
]

print("Possible validation files:")

for name in validation_files:
    print("-", name)

Possible validation files:
- crop_water_parameter.csv
- irrigation_method.csv
- literature_environmental_measurement.csv
- literature_trial_observation.csv
- literature_trial_result.csv
- rice_daily_water_demand.csv
- water_scenario.csv
- water_scenario_by_variety.csv
- water_scenario_by_variety_clay.csv
- water_scenario_by_variety_v3.csv


In [18]:
for filename in validation_files:
    full_path = raw_path + filename

    try:
        df = pd.read_csv(full_path)

        print("\n====================")
        print(filename)
        print("Shape:", df.shape)
        print("Columns:")
        print(df.columns.tolist())

    except Exception as e:
        print(
            "\nCould not read:",
            filename,
            "|",
            e
        )


crop_water_parameter.csv
Shape: (10, 13)
Columns:
['parameter_id', 'crop', 'variety_id', 'growth_stage', 'kc', 'parameter_name', 'parameter_value', 'unit', 'valid_conditions', 'source_url', 'data_status', 'value_min', 'value_max']

irrigation_method.csv
Shape: (2, 4)
Columns:
['method_id', 'name_en', 'name_ar', 'notes']

literature_environmental_measurement.csv
Shape: (104, 14)
Columns:
['measurement_id', 'study_id', 'sample_kind', 'sample_label', 'sample_year', 'treatment_code', 'depth_top_cm', 'depth_bottom_cm', 'parameter_name', 'parameter_value', 'unit', 'source_locator', 'data_status', 'source_url']

literature_trial_observation.csv
Shape: (688, 16)
Columns:
['observation_id', 'study_id', 'study_year', 'observation_period', 'preceding_crop', 'puddling', 'depth_top_cm', 'depth_bottom_cm', 'metric_name', 'metric_value', 'unit', 'aggregation_level', 'replicate_count', 'source_locator', 'data_status', 'source_url']

literature_trial_result.csv
Shape: (470, 13)
Columns:
['result_id', 

In [19]:
irrigation_method = pd.read_csv(
    raw_path + "irrigation_method.csv"
)

water_scenario = pd.read_csv(
    raw_path + "water_scenario.csv"
)

water_scenario_clay = pd.read_csv(
    raw_path + "water_scenario_by_variety_clay.csv"
)

water_scenario_v3 = pd.read_csv(
    raw_path + "water_scenario_by_variety_v3.csv"
)

trial_result = pd.read_csv(
    raw_path + "literature_trial_result.csv"
)

trial_observation = pd.read_csv(
    raw_path + "literature_trial_observation.csv"
)

environmental_measurement = pd.read_csv(
    raw_path + "literature_environmental_measurement.csv"
)

print("Irrigation methods:", irrigation_method.shape)
print("Water scenario:", water_scenario.shape)
print("Clay scenario:", water_scenario_clay.shape)
print("V3 scenario:", water_scenario_v3.shape)
print("Trial results:", trial_result.shape)
print("Trial observations:", trial_observation.shape)
print("Environmental measurements:", environmental_measurement.shape)

Irrigation methods: (2, 4)
Water scenario: (266, 16)
Clay scenario: (2660, 18)
V3 scenario: (9044, 22)
Trial results: (470, 13)
Trial observations: (688, 16)
Environmental measurements: (104, 14)


In [20]:
display(irrigation_method)

,method_id,name_en,name_ar,notes
0,1,continuous_flooding,غمر تقليدي,NaN
1,2,alternate_wetting_drying,ري متقطع,NaN


In [21]:
print("Existing water-scenario statuses:")

print(
    water_scenario[
        "actual_irrigation_status"
    ].value_counts(dropna=False)
)

print("\nMethod status:")

print(
    water_scenario[
        "method_status"
    ].value_counts(dropna=False)
)

Existing water-scenario statuses:
actual_irrigation_status
NOT_MEASURED    266
Name: count, dtype: int64

Method status:
method_status
SCENARIO_ESTIMATE_NOT_FIELD_VALIDATED    266
Name: count, dtype: int64


In [22]:
#soil/clay seepage logic
clay_seepage_summary = (
    water_scenario_clay[
        [
            "seepage_class",
            "seepage_low_mm_day",
            "seepage_high_mm_day"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "seepage_low_mm_day",
            "seepage_high_mm_day"
        ]
    )
)

clay_seepage_summary

,seepage_class,seepage_low_mm_day,seepage_high_mm_day
0,clay_ge_40,1.0,3.0
160,clay_25_40,2.0,5.0
10,no_clay_data_default_4_8,4.0,8.0
90,clay_lt_25,4.0,8.0


In [23]:
print(
    water_scenario_clay[
        "seepage_class"
    ].value_counts(dropna=False))

seepage_class
clay_lt_25                  1260
clay_25_40                   700
clay_ge_40                   560
no_clay_data_default_4_8     140
Name: count, dtype: int64


In [24]:
#Merge old and new results
scenario_comparison = irrigation.merge(
    water_scenario[
        [
            "point_id",
            "season_year",
            "estimated_etc_season_mm",
            "rainfall_season_mm",
            "scenario_low_depth_mm",
            "scenario_high_depth_mm",
            "scenario_low_m3_per_feddan",
            "scenario_high_m3_per_feddan",
            "actual_irrigation_status",
            "method_status"
        ]
    ],
    on=[
        "point_id",
        "season_year"
    ],
    how="left",
    validate="one_to_one"
)

print(
    "Missing comparison rows:",
    scenario_comparison[
        "scenario_low_depth_mm"
    ].isna().sum()
)

print(
    "Comparison rows:",
    len(scenario_comparison)
)

Missing comparison rows: 0
Comparison rows: 266


In [25]:
# Calculate differences
scenario_comparison[
    "low_depth_difference_mm"
] = (
    scenario_comparison[
        "net_irrigation_low_mm"
    ]
    -
    scenario_comparison[
        "scenario_low_depth_mm"
    ]
)

scenario_comparison[
    "high_depth_difference_mm"
] = (
    scenario_comparison[
        "net_irrigation_high_mm"
    ]
    -
    scenario_comparison[
        "scenario_high_depth_mm"
    ]
)

scenario_comparison[
    "low_volume_difference_m3_feddan"
] = (
    scenario_comparison[
        "net_low_m3_per_feddan"
    ]
    -
    scenario_comparison[
        "scenario_low_m3_per_feddan"
    ]
)

scenario_comparison[
    "high_volume_difference_m3_feddan"
] = (
    scenario_comparison[
        "net_high_m3_per_feddan"
    ]
    -
    scenario_comparison[
        "scenario_high_m3_per_feddan"
    ]
)

scenario_comparison[
    [
        "low_depth_difference_mm",
        "high_depth_difference_mm",
        "low_volume_difference_m3_feddan",
        "high_volume_difference_m3_feddan"
    ]
].describe()

,low_depth_difference_mm,high_depth_difference_mm,low_volume_difference_m3_feddan,high_volume_difference_m3_feddan
count,266.000000,266.000000,266.000000,266.000000
mean,-205.440083,-280.012487,-862.844062,-1176.048159
std,18.855743,18.308518,79.194310,76.895906
min,-223.535460,-296.522018,-938.844931,-1245.388475
25%,-218.651822,-292.780776,-918.313652,-1229.665261
50%,-212.893956,-286.559083,-894.132616,-1203.554148
75%,-201.358921,-276.340814,-845.712470,-1160.621417
max,-154.227729,-229.667903,-647.744464,-964.593192


In [26]:
# ETc Consistency
scenario_comparison[
    "etc_difference_mm"
] = (
    scenario_comparison[
        "ETc_low_mm"
    ]
    -
    scenario_comparison[
        "estimated_etc_season_mm"
    ]
)

scenario_comparison[
    [
        "ETc_low_mm",
        "estimated_etc_season_mm",
        "etc_difference_mm"
    ]
].describe()

,ETc_low_mm,estimated_etc_season_mm,etc_difference_mm
count,266.000000,266.000000,266.000000
mean,1115.678323,1234.995000,-119.316677
std,83.550181,82.700679,3.955432
min,1012.330225,1132.510000,-124.885460
25%,1041.657613,1160.440000,-122.318820
50%,1063.440367,1185.680000,-120.377651
75%,1203.676894,1324.360000,-116.592702
max,1234.911432,1356.120000,-106.299641


In [27]:
# Find water-related trial-result metrics
trial_water_metrics = sorted(
    trial_result.loc[
        trial_result[
            "metric_name"
        ]
        .astype(str)
        .str.contains(
            "water|irrig|consumption|requirement|use",
            case=False,
            regex=True,
            na=False
        ),
        "metric_name"
    ]
    .dropna()
    .unique()
)

print("Water-related trial result metrics:")

for metric in trial_water_metrics:
    print("-", metric)

Water-related trial result metrics:
- irrigation_water_applied
- water_productivity


In [28]:
# Check trial observations
observation_water_metrics = sorted(
    trial_observation.loc[
        trial_observation[
            "metric_name"
        ]
        .astype(str)
        .str.contains(
            "water|irrig|consumption|requirement|use",
            case=False,
            regex=True,
            na=False
        ),
        "metric_name"
    ]
    .dropna()
    .unique()
)

print("Water-related observation metrics:")

for metric in observation_water_metrics:
    print("-", metric)

Water-related observation metrics:
- available_water
- crop_water_use_efficiency_kg_m3
- irrigation_after_transplant_mm
- irrigation_applied_mm_season
- irrigation_water_applied
- irrigation_water_productivity
- irrigation_water_productivity_kg_m3
- nursery_water_mm
- seasonal_irrigation_applied
- total_irrigation_water
- total_water_applied
- water_productivity
- water_saved
- water_saving
- water_stable_aggregates_gt_0_25mm
- water_use_efficiency


In [29]:
# Check environmental measurements
environment_water_metrics = sorted(
    environmental_measurement.loc[
        environmental_measurement[
            "parameter_name"
        ]
        .astype(str)
        .str.contains(
            "water|irrig|moisture|percol|seep",
            case=False,
            regex=True,
            na=False
        ),
        "parameter_name"
    ]
    .dropna()
    .unique()
)

print("Relevant environmental parameters:")

for parameter in environment_water_metrics:
    print("-", parameter)

Relevant environmental parameters:


In [30]:
# Literature irrigation benchmark

trial_result_irrigation = trial_result[
    trial_result["metric_name"].isin([
        "irrigation_water_applied"
    ])
].copy()

trial_result_irrigation["source_table"] = (
    "literature_trial_result"
)

trial_result_irrigation = (
    trial_result_irrigation.rename(
        columns={
            "metric_value": "value",
            "metric_name": "metric"
        }
    )
)

observation_irrigation = trial_observation[
    trial_observation["metric_name"].isin([
        "irrigation_applied_mm_season",
        "seasonal_irrigation_applied",
        "total_irrigation_water",
        "total_water_applied"
    ])
].copy()

observation_irrigation["source_table"] = (
    "literature_trial_observation"
)

observation_irrigation = (
    observation_irrigation.rename(
        columns={
            "metric_value": "value",
            "metric_name": "metric"
        }
    )
)

literature_irrigation = pd.concat(
    [
        trial_result_irrigation[
            [
                "source_table",
                "study_id",
                "metric",
                "value",
                "unit",
                "data_status"
            ]
        ],
        observation_irrigation[
            [
                "source_table",
                "study_id",
                "metric",
                "value",
                "unit",
                "data_status"
            ]
        ]
    ],
    ignore_index=True
)

print(
    "Literature irrigation rows:",
    len(literature_irrigation)
)

display(
    literature_irrigation[
        [
            "metric",
            "unit",
            "data_status"
        ]
    ]
    .value_counts()
    .reset_index(
        name="count"
    )
)

Literature irrigation rows: 65


,metric,unit,data_status,count
0,seasonal_irrigation_applied,m3/feddan,PUBLISHED_SAKHA_STATION_TREATMENT_MEAN_NOT_FAR...,36
1,irrigation_water_applied,m3_per_feddan,table_transcription,12
2,total_water_applied,m3/ha,PUBLISHED_EL_KARADA_STATION_TREATMENT_MEAN_NOT...,8
3,total_irrigation_water,m3/ha,PUBLISHED_SAKHA_STATION_TREATMENT_MEAN_NOT_FAR...,6
4,irrigation_applied_mm_season,mm/season,PUBLISHED_SAKHA_STATION_RESULT_NOT_FARM_LABEL,3


In [31]:
# Convert literature values to mm/season
def irrigation_value_to_mm(value, unit):

    if pd.isna(value) or pd.isna(unit):
        return np.nan

    value = float(value)

    u = (
        str(unit)
        .lower()
        .replace("³", "3")
        .replace(" ", "")
        .replace("per", "/")
    )

    # Already depth
    if "mm" in u:
        return value

    # Cubic metres per feddan
    if (
        "m3" in u
        and (
            "feddan" in u
            or "fed" in u
        )
    ):
        return value / 4.2

    # Cubic metres per hectare
    if (
        "m3" in u
        and (
            "/ha" in u
            or "ha-1" in u
            or "hectare" in u
        )
    ):
        return value / 10

    return np.nan


literature_irrigation[
    "irrigation_mm"
] = literature_irrigation.apply(
    lambda row:
    irrigation_value_to_mm(
        row["value"],
        row["unit"]
    ),
    axis=1
)

print(
    "Converted literature rows:",
    literature_irrigation[
        "irrigation_mm"
    ].notna().sum()
)

print(
    "Unconverted rows:",
    literature_irrigation[
        "irrigation_mm"
    ].isna().sum()
)

display(
    literature_irrigation[
        literature_irrigation[
            "irrigation_mm"
        ].isna()
    ][
        [
            "metric",
            "value",
            "unit"
        ]
    ].drop_duplicates()
)

Converted literature rows: 65
Unconverted rows: 0


,metric,value,unit


In [32]:
# Literature irrigation summary
literature_mm = literature_irrigation[
    literature_irrigation[
        "irrigation_mm"
    ].notna()
].copy()

literature_mm = literature_mm[
    literature_mm[
        "irrigation_mm"
    ] > 0
].copy()

literature_summary = (
    literature_mm[
        "irrigation_mm"
    ]
    .describe(
        percentiles=[
            0.05,
            0.25,
            0.50,
            0.75,
            0.95
        ]
    )
)

literature_summary

,irrigation_mm
count,65.000000
mean,1162.081319
std,294.457386
min,640.476190
5%,678.571429
25%,943.500000
50%,1176.100000
75%,1414.100000
95%,1570.523810
max,1614.285714


In [33]:
display(
    literature_mm.groupby(
        [
            "metric",
            "unit"])["irrigation_mm"].agg([
            "count",
            "min",
            "median",
            "mean",
            "max"]).reset_index())

,metric,unit,count,min,median,mean,max
0,irrigation_applied_mm_season,mm/season,3,1176.100000,1208.900000,1206.200000,1233.600000
1,irrigation_water_applied,m3_per_feddan,12,923.333333,1519.047619,1363.452381,1575.476190
2,seasonal_irrigation_applied,m3/feddan,36,640.476190,988.690476,1049.107143,1614.285714
3,total_irrigation_water,m3/ha,6,943.500000,1046.800000,1094.866667,1300.500000
4,total_water_applied,m3/ha,8,1346.900000,1389.850000,1402.275000,1484.200000


In [34]:
literature_low_mm = (
    literature_mm[
        "irrigation_mm"
    ].quantile(0.05)
)

literature_high_mm = (
    literature_mm[
        "irrigation_mm"
    ].quantile(0.95)
)

literature_median_mm = (
    literature_mm[
        "irrigation_mm"
    ].median()
)

print(
    "Literature 5th percentile:",
    literature_low_mm
)

print(
    "Literature median:",
    literature_median_mm
)

print(
    "Literature 95th percentile:",
    literature_high_mm)

Literature 5th percentile: 678.5714285714286
Literature median: 1176.1
Literature 95th percentile: 1570.5238095238094


In [35]:
irrigation["literature_range_overlap"] = ((irrigation["net_irrigation_low_mm"] <= literature_high_mm) & (irrigation["net_irrigation_high_mm"] >= literature_low_mm))

print(irrigation["literature_range_overlap"].value_counts())

print("Percent overlapping literature range:")

print(irrigation["literature_range_overlap"].mean() * 100)

literature_range_overlap
False    266
Name: count, dtype: int64
Percent overlapping literature range:
0.0


In [36]:
soil_clay = soil[
    [
        "point_id",
        "clay_pct"]].copy()

irrigation = irrigation.merge(
    soil_clay,
    on="point_id",
    how="left",
    validate="many_to_one")

print("Rows without usable soil:", irrigation["clay_pct"].isna().sum())

Rows without usable soil: 14


In [37]:
conditions = [
    irrigation[
        "clay_pct"
    ].ge(40),

    irrigation[
        "clay_pct"
    ].between(
        25,
        40,
        inclusive="left"
    ),

    irrigation[
        "clay_pct"
    ].lt(25)
]

seepage_classes = [
    "clay_ge_40",
    "clay_25_40",
    "clay_lt_25"
]

irrigation[
    "soil_seepage_class"
] = np.select(
    conditions,
    seepage_classes,
    default="no_clay_data_default_4_8"
)

low_map = {
    "clay_ge_40": 1,
    "clay_25_40": 2,
    "clay_lt_25": 4,
    "no_clay_data_default_4_8": 4
}

high_map = {
    "clay_ge_40": 3,
    "clay_25_40": 5,
    "clay_lt_25": 8,
    "no_clay_data_default_4_8": 8
}

irrigation[
    "soil_perc_low_mm_day"
] = irrigation[
    "soil_seepage_class"
].map(low_map)

irrigation[
    "soil_perc_high_mm_day"
] = irrigation[
    "soil_seepage_class"
].map(high_map)

irrigation[
    "soil_seepage_class"
].value_counts()

,count
soil_seepage_class,
clay_lt_25,126
clay_25_40,70
clay_ge_40,56
no_clay_data_default_4_8,14


In [38]:
irrigation[
    "soil_perc_low_mm"
] = (
    irrigation[
        "season_days"
    ]
    *
    irrigation[
        "soil_perc_low_mm_day"
    ]
)

irrigation[
    "soil_perc_high_mm"
] = (
    irrigation[
        "season_days"
    ]
    *
    irrigation[
        "soil_perc_high_mm_day"
    ]
)

irrigation[
    "soil_adjusted_low_mm"
] = (
    irrigation[
        "ETc_low_mm"
    ]
    +
    irrigation[
        "SAT_mm"
    ]
    +
    irrigation[
        "soil_perc_low_mm"
    ]
    +
    irrigation[
        "WL_mm"
    ]
    -
    irrigation[
        "effective_rain_mm"
    ]
)

irrigation[
    "soil_adjusted_high_mm"
] = (
    irrigation[
        "ETc_high_mm"
    ]
    +
    irrigation[
        "SAT_mm"
    ]
    +
    irrigation[
        "soil_perc_high_mm"
    ]
    +
    irrigation[
        "WL_mm"
    ]
    -
    irrigation[
        "effective_rain_mm"
    ]
)

irrigation[
    "soil_adjusted_low_mm"
] = irrigation[
    "soil_adjusted_low_mm"
].clip(lower=0)

irrigation[
    "soil_adjusted_high_mm"
] = irrigation[
    "soil_adjusted_high_mm"
].clip(lower=0)

In [39]:
irrigation[
    "soil_adjusted_low_m3_feddan"
] = (
    irrigation[
        "soil_adjusted_low_mm"
    ]
    * 4.2
)

irrigation[
    "soil_adjusted_high_m3_feddan"
] = (
    irrigation[
        "soil_adjusted_high_mm"
    ]
    * 4.2
)

irrigation[
    [
        "soil_adjusted_low_mm",
        "soil_adjusted_high_mm",
        "soil_adjusted_low_m3_feddan",
        "soil_adjusted_high_m3_feddan"
    ]
].describe()

,soil_adjusted_low_mm,soil_adjusted_high_mm,soil_adjusted_low_m3_feddan,soil_adjusted_high_m3_feddan
count,266.000000,266.000000,266.000000,266.000000
mean,1838.791872,2361.587889,7722.925863,9918.669134
std,189.590505,297.653523,796.280123,1250.144798
min,1612.370363,1946.841118,6771.955525,8176.732697
25%,1654.165242,2097.061014,6947.494016,8807.656257
50%,1913.165480,2537.176554,8035.295017,10656.141528
75%,1961.417864,2584.916703,8237.955027,10856.650154
max,2134.829520,2761.289702,8966.283985,11597.416748


In [40]:
irrigation[
    "stored_soil_water_adjustment_mm"
] = 0.0

irrigation[
    "stored_soil_water_status"
] = (
    "NOT_APPLIED_NO_INITIAL_SOIL_MOISTURE_MEASUREMENT"
)

In [41]:
efficiency_scenarios = [
    0.60,
    0.70,
    0.80
]

for efficiency in efficiency_scenarios:

    pct = int(
        efficiency * 100
    )

    irrigation[
        f"gross_low_mm_eff_{pct}"
    ] = (
        irrigation[
            "net_irrigation_low_mm"
        ]
        /
        efficiency
    )

    irrigation[
        f"gross_high_mm_eff_{pct}"
    ] = (
        irrigation[
            "net_irrigation_high_mm"
        ]
        /
        efficiency
    )

    irrigation[
        f"gross_low_m3_feddan_eff_{pct}"
    ] = (
        irrigation[
            f"gross_low_mm_eff_{pct}"
        ]
        * 4.2
    )

    irrigation[
        f"gross_high_m3_feddan_eff_{pct}"
    ] = (
        irrigation[
            f"gross_high_mm_eff_{pct}"
        ]
        * 4.2
    )

In [42]:
# Review gross requirements
irrigation[
    [
        "gross_low_m3_feddan_eff_60",
        "gross_high_m3_feddan_eff_60",
        "gross_low_m3_feddan_eff_70",
        "gross_high_m3_feddan_eff_70",
        "gross_low_m3_feddan_eff_80",
        "gross_high_m3_feddan_eff_80"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
gross_low_m3_feddan_eff_60,266.0,14087.332579,592.416385,13386.311578,13567.780442,13744.082570,14725.738261,14944.380027
gross_high_m3_feddan_eff_60,266.0,18465.325750,598.735483,17756.146231,17939.694928,18124.698195,19113.408362,19329.714951
gross_low_m3_feddan_eff_70,266.0,12074.856496,507.785472,11473.981353,11629.526093,11780.642203,12622.061367,12809.468595
gross_high_m3_feddan_eff_70,266.0,15827.422072,513.201842,15219.553912,15376.881367,15535.455596,16382.921453,16568.327101
gross_low_m3_feddan_eff_80,266.0,10565.499434,444.312288,10039.733684,10175.835332,10308.061928,11044.303696,11208.285020
gross_high_m3_feddan_eff_80,266.0,13848.994313,449.051612,13317.109673,13454.771196,13593.523646,14335.056271,14497.286214


In [43]:
irrigation[
    "actual_irrigation_status"
] = "NOT_FIELD_MEASURED"

irrigation[
    "main_model_status"
] = (
    "FAO_BASED_SCENARIO_NOT_FIELD_VALIDATED"
)

irrigation[
    "soil_sensitivity_status"
] = (
    "PROJECT_SOIL_SENSITIVITY_NOT_FIELD_VALIDATED"
)

irrigation[
    "efficiency_status"
] = (
    "SENSITIVITY_ANALYSIS_NOT_MEASURED"
)

In [44]:
# summary
validation_summary = pd.DataFrame(
    {
        "validation_check": [
            "Existing scenario comparison",
            "ETc comparison",
            "Literature irrigation benchmark",
            "Environmental water validation",
            "Irrigation efficiency",
            "Stored soil water"
        ],

        "result": [
            (
                "New model lower than previous "
                "scenario estimates"
            ),

            (
                "New stage-based ETc lower than "
                "previous ETc estimate"
            ),

            (
                f"Literature benchmark: "
                f"{literature_low_mm:.1f} to "
                f"{literature_high_mm:.1f} mm "
                f"(5th-95th percentile)"
            ),

            (
                "No relevant water/percolation "
                "measurements available"
            ),

            (
                "60%, 70%, and 80% sensitivity "
                "scenarios; no measured efficiency"
            ),

            (
                "Not deducted because actual "
                "initial soil moisture is unavailable"
            )
        ],

        "status": [
            "CONSISTENCY_CHECK_ONLY",
            "CONSISTENCY_CHECK_ONLY",
            "INDEPENDENT_LITERATURE_BENCHMARK",
            "NOT_AVAILABLE",
            "ASSUMPTION_SENSITIVITY",
            "NOT_AVAILABLE"
        ]
    }
)

validation_summary

,validation_check,result,status
0,Existing scenario comparison,New model lower than previous scenario estimates,CONSISTENCY_CHECK_ONLY
1,ETc comparison,New stage-based ETc lower than previous ETc es...,CONSISTENCY_CHECK_ONLY
2,Literature irrigation benchmark,Literature benchmark: 678.6 to 1570.5 mm (5th-...,INDEPENDENT_LITERATURE_BENCHMARK
3,Environmental water validation,No relevant water/percolation measurements ava...,NOT_AVAILABLE
4,Irrigation efficiency,"60%, 70%, and 80% sensitivity scenarios; no me...",ASSUMPTION_SENSITIVITY
5,Stored soil water,Not deducted because actual initial soil moist...,NOT_AVAILABLE


In [45]:
print(
    "Final seasonal rows:",
    len(irrigation)
)

print(
    "Missing main low requirement:",
    irrigation[
        "net_irrigation_low_mm"
    ].isna().sum()
)

print(
    "Missing main high requirement:",
    irrigation[
        "net_irrigation_high_mm"
    ].isna().sum()
)

print(
    "Negative main requirement:",
    (
        irrigation[
            "net_irrigation_low_mm"
        ] < 0
    ).sum()
    +
    (
        irrigation[
            "net_irrigation_high_mm"
        ] < 0
    ).sum()
)

print(
    "Low > high:",
    (
        irrigation[
            "net_irrigation_low_mm"
        ]
        >
        irrigation[
            "net_irrigation_high_mm"
        ]
    ).sum()
)

print(
    "Literature overlap rate:",
    irrigation[
        "literature_range_overlap"
    ].mean()
    * 100,
    "%"
)

Final seasonal rows: 266
Missing main low requirement: 0
Missing main high requirement: 0
Negative main requirement: 0
Low > high: 0
Literature overlap rate: 0.0 %


In [46]:
irrigation_output_path = (
    processed_path
    + "irrigation_requirement_seasonal.csv"
)

validation_output_path = (
    processed_path
    + "irrigation_validation_summary.csv"
)

literature_output_path = (
    processed_path
    + "literature_irrigation_benchmark.csv"
)

irrigation.to_csv(
    irrigation_output_path,
    index=False
)

validation_summary.to_csv(
    validation_output_path,
    index=False
)

literature_mm.to_csv(
    literature_output_path,
    index=False
)

print(
    "Saved:",
    irrigation_output_path
)

print(
    "Saved:",
    validation_output_path
)

print(
    "Saved:",
    literature_output_path
)

Saved: /content/Smart-Agricultural-Water-Management/weather_soil_water_modeling/data/processed/irrigation_requirement_seasonal.csv
Saved: /content/Smart-Agricultural-Water-Management/weather_soil_water_modeling/data/processed/irrigation_validation_summary.csv
Saved: /content/Smart-Agricultural-Water-Management/weather_soil_water_modeling/data/processed/literature_irrigation_benchmark.csv


In [47]:
print(
    "Irrigation file exists:",
    os.path.exists(
        irrigation_output_path
    )
)

print(
    "Validation file exists:",
    os.path.exists(
        validation_output_path
    )
)

print(
    "Literature file exists:",
    os.path.exists(
        literature_output_path
    )
)

print(
    "Irrigation rows:",
    len(irrigation)
)

Irrigation file exists: True
Validation file exists: True
Literature file exists: True
Irrigation rows: 266


In [49]:
# Final validation interpretation

model_low_min = irrigation["net_irrigation_low_mm"].min()
model_high_max = irrigation["net_irrigation_high_mm"].max()

validation_interpretation = (
    "The FAO-based modeled irrigation requirement does not overlap "
    "with the 5th–95th percentile range of the available literature "
    "irrigation measurements. This indicates that the model produces "
    "higher seasonal water requirements than the experimental literature "
    "benchmark. The discrepancy may reflect differences in irrigation "
    "method, experimental treatment, seepage/percolation assumptions, "
    "soil conditions, and inclusion of land preparation and standing-water "
    "requirements. Therefore, the results should be treated as scenario-based "
    "water requirement estimates rather than field-validated irrigation demand."
)

print(validation_interpretation)

print("\nModel range:")
print(
    round(model_low_min, 1),
    "to",
    round(model_high_max, 1),
    "mm/season"
)

print("\nLiterature benchmark:")
print(
    round(literature_low_mm, 1),
    "to",
    round(literature_high_mm, 1),
    "mm/season"
)

The FAO-based modeled irrigation requirement does not overlap with the 5th–95th percentile range of the available literature irrigation measurements. This indicates that the model produces higher seasonal water requirements than the experimental literature benchmark. The discrepancy may reflect differences in irrigation method, experimental treatment, seepage/percolation assumptions, soil conditions, and inclusion of land preparation and standing-water requirements. Therefore, the results should be treated as scenario-based water requirement estimates rather than field-validated irrigation demand.

Model range:
1912.3 to 2761.4 mm/season

Literature benchmark:
678.6 to 1570.5 mm/season


In [50]:
validation_summary.loc[
    validation_summary["validation_check"]
    == "Literature irrigation benchmark",
    "status"
] = "BENCHMARK_DISCREPANCY_REQUIRES_CAUTION"

validation_summary.loc[
    validation_summary["validation_check"]
    == "Literature irrigation benchmark",
    "result"
] = (
    f"Model range does not overlap literature benchmark "
    f"({literature_low_mm:.1f}–{literature_high_mm:.1f} mm). "
    f"Results are scenario estimates and are not field validated."
)

validation_summary

,validation_check,result,status
0,Existing scenario comparison,New model lower than previous scenario estimates,CONSISTENCY_CHECK_ONLY
1,ETc comparison,New stage-based ETc lower than previous ETc es...,CONSISTENCY_CHECK_ONLY
2,Literature irrigation benchmark,Model range does not overlap literature benchm...,BENCHMARK_DISCREPANCY_REQUIRES_CAUTION
3,Environmental water validation,No relevant water/percolation measurements ava...,NOT_AVAILABLE
4,Irrigation efficiency,"60%, 70%, and 80% sensitivity scenarios; no me...",ASSUMPTION_SENSITIVITY
5,Stored soil water,Not deducted because actual initial soil moist...,NOT_AVAILABLE


In [54]:
validation_summary.to_csv(
    validation_output_path,
    index=False
)

print("Final validation summary saved.")
print(validation_output_path)

Final validation summary saved.
/content/Smart-Agricultural-Water-Management/weather_soil_water_modeling/data/processed/irrigation_validation_summary.csv


In [55]:
from google.colab import files

files.download(
    irrigation_output_path
)

files.download(
    validation_output_path
)

files.download(
    literature_output_path
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>